<a href="https://colab.research.google.com/github/mikysetiawan/MachineLearningExpert/blob/master/AlphaZero_TicTacToe_MCTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
np.__version__

import math

In [5]:
class TicTacToe:
  def __init__(self):
    self.row_count = 3
    self.column_count = 3
    self.action_size = self.row_count * self.column_count

  def get_initial_state(self):
    return np.zeros((self.row_count, self.column_count))

  def get_next_state(self, state, action, player):
    row = action // self.column_count # (Floor Division rounds down), unlike standard division (/) that may resulting decimal, it automatically round
    column = action % self.column_count
    state[row, column] = player
    return state

  def get_valid_moves(self, state):
    return (state.reshape(-1) == 0).astype(np.uint8)

  def check_win(self, state, action):
    if action == None:
      return False

    row = action // self.column_count
    column = action % self.column_count
    player = state[row, column]

    return (
      np.sum(state[row, :]) == player * self.column_count # check if all column is true
      or np.sum(state[:, column]) == player * self.row_count # check if all row is true
      or np.sum(np.diag(state)) == player * self.row_count # check if all diagonal is true
      or np.sum(np.diag(np.flip(state, axis=0))) == player * self.row_count # check if flip diagonal is true
    )

  def get_value_and_terimanted(self, state, action):
    if self.check_win(state, action):
      return 1, True
    if np.sum(self.get_valid_moves(state)) == 0:
      return 0, True
    return 0, False

  def get_opponent(self, player):
    return -player

  def get_opponent_value(self, value):
    return -value

  def change_perspective(self, state, player):
    return state * player

In [11]:
class Node:
  def __init__(self, game, args, state, parent=None, action_taken=None):
    self.state = state
    self.game = game
    self.args = args
    self.parent = parent
    self.action_taken = action_taken

    self.children = []
    self.expandable_moves = game.get_valid_moves(state)

    self.visit_count = 0
    self.value_sum = 0

  def is_fully_expanded(self):
    return np.sum(self.expandable_moves) == 0 and len(self.children) > 0

  def select(self):
    best_child = None
    best_ucb = -np.inf

    for child in self.children:
      ucb = self.get_ucb(child)
      if ucb > best_ucb:
        best_child = child
        best_ucb = ucb
    return best_child

  def get_ucb(self, child):
    q_value = 1 - ((child.value_sum / child.visit_count) + 1) / 2
    # + 1) / 2: to shifts and scales the values from [-1, 1] to a clean [0, 1] range (where 0 is a total loss and 1 is a perfect win)
    # "1 - (...): This inverts the score. Because games alternate turns (gantian), a position that is excellent for the opponent (the child) is terrible for the current player (the parent). Inverting it ensures the parent selects a move that is good for themselves, not the opponent.
    return q_value + self.args["C"] * math.sqrt(math.log(self.visit_count) / child.visit_count)

  def expand(self):
    action = np.random.choice(np.where(self.expandable_moves == 1)[0])
    self.expandable_moves[action] = 0

    child_state = self.state.copy()
    child_state = self.game.get_next_state(child_state, action, 1)
    child_state = self.game.change_perspective(child_state, player=-1)

    child = Node(self.game, self.args, child_state, self, action)
    self.children.append(child)
    return child

  def simulate(self):
    value, is_terminate = self.game.get_value_and_terimanted(self.state, self.action_taken)
    value = self.game.get_opponent_value(value)

    if is_terminate:
      return value

    rollout_state = self.state.copy()
    rollout_player = 1
    while True:
      valid_moves = self.game.get_valid_moves(rollout_state)
      action = np.random.choice(np.where(valid_moves == 1)[0])
      rollout_state = self.game.get_next_state(rollout_state, action, rollout_player)

      value, is_terminate = self.game.get_value_and_terimanted(rollout_state, action)

      if is_terminate:
        if rollout_player == -1:
          value = self.game.get_opponent_value(value)
        return value

      rollout_player = self.game.get_opponent(rollout_player)

  def backpropagate(self, value):
    self.value_sum += value
    self.visit_count += 1

    value = self.game.get_opponent_value(value)
    if self.parent is not None:
      self.parent.backpropagate(value)

class MCTS:
  def __init__(self, game, args):
    self.game = game
    self.args = args

  def search(self, state):
    # define root
    root = Node(self.game, self.args, state)

    for search in range(self.args["num_searches"]):
      node = root

      # SELECTION
      while node.is_fully_expanded():
        node = node.select()

      # check is terminated or not
      value, is_terminate = self.game.get_value_and_terimanted(node.state, node.action_taken)
      value = self.game.get_opponent(value)

      if not is_terminate:
        # EXPANSION
        node = node.expand()

        # SIMULATION
        value = node.simulate()

      # BACKPROPAGATION
      node.backpropagate(value)

    # return visit_counts
    action_probs = np.zeros(self.game.action_size)
    for child in root.children:
      action_probs[child.action_taken] = child.visit_count
    action_probs /= np.sum(action_probs)
    return action_probs

In [12]:
tictactoe = TicTacToe()
player = 1

args = {
    'C': 1.41,
    'num_searches': 1000
}

mcts = MCTS(game=tictactoe, args=args)
state = tictactoe.get_initial_state()

while True:
  print(state)

  if player == 1:
    valid_moves = tictactoe.get_valid_moves(state)
    print("valid_moves", [i for i in range(tictactoe.action_size) if valid_moves[i] == 1])

    action = int(input(f"{player}:"))

    if valid_moves[action] == 0:
      print("Invalid move!")
      continue
  else:
    neutral_state = tictactoe.change_perspective(state, player)
    mcts_probs = mcts.search(state)
    action = np.argmax(mcts_probs)


  state = tictactoe.get_next_state(state, action, player)

  value, is_terminate = tictactoe.get_value_and_terimanted(state, action)
  if is_terminate:
    print(state)
    if value == 1:
      print(player, "won")
    else:
      print("Draw")
    break

  player = tictactoe.get_opponent(player)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
valid_moves [0, 1, 2, 3, 4, 5, 6, 7, 8]
1:4
[[0. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]]
[[ 0.  0.  0.]
 [ 0.  1.  0.]
 [ 0.  0. -1.]]
valid_moves [0, 1, 2, 3, 5, 6, 7]
1:5
[[ 0.  0.  0.]
 [ 0.  1.  1.]
 [ 0.  0. -1.]]
[[ 0.  0.  0.]
 [-1.  1.  1.]
 [ 0.  0. -1.]]
valid_moves [0, 1, 2, 6, 7]
1:6
[[ 0.  0.  0.]
 [-1.  1.  1.]
 [ 1.  0. -1.]]
[[ 0.  0. -1.]
 [-1.  1.  1.]
 [ 1.  0. -1.]]
valid_moves [0, 1, 7]
1:7
[[ 0.  0. -1.]
 [-1.  1.  1.]
 [ 1.  1. -1.]]
[[ 0. -1. -1.]
 [-1.  1.  1.]
 [ 1.  1. -1.]]
valid_moves [0]
1:0
[[ 1. -1. -1.]
 [-1.  1.  1.]
 [ 1.  1. -1.]]
Draw
